
<div style="text-align: right;">
  <a href="../../Files/invoke_model.ipynb"
     download="invoke_model.ipynb"
     class="md-button md-button--primary">
    ⬇️ Download Notebook
  </a>


## Flow / Sequence

**Importing necessary libraries**

In [1]:
import os
import base64
import requests
import time
import json

### API Configuration

This dictionary holds the configuration for API endpoints and base URLs. It includes :

base_url: The base URL for the API, which can be overridden by an environment variable.
base_oauth_endpoint: The endpoint for OAuth authentication.
endpoints: Specific endpoints for invoking models, checking job status, retrieving data, and obtaining OAuth tokens.

In [ ]:
API_CONFIG = {
    "base_url": os.getenv("API_BASE_URL", ""),
    "base_oauth_endpoint":"",
    "endpoints": {
        "invoke": "/models/bedrock", 
        "status": "/jobs",
        "data": "/jobs",
        "oauth": "/oauth2/token"
    },
   
}

### Credentials 

This dictionary stores the credentials required for OAuth authentication. These should be securely stored and retrieved, typically from environment variables or a secure vault.

In [ ]:
creds = {
    "client_id": os.getenv("CLIENT_ID", ""),
    "client_secret": os.getenv("CLIENT_SECRET", ""),
    "api_key": os.getenv("API_KEY", "")
}


### Function to Obtain OAuth Token

This function handles the OAuth authentication process:

-  Constructs the OAuth URL.
-  Encodes client credentials using Base64.
-  Sets up headers and request body for the POST request.
-  Sends the request to obtain the OAuth token.
-  Checks for errors and extracts the token from the response.

In [4]:
def get_oauth_token():
    """
    Function to obtain OAuth token using client credentials.
    
    Returns:
        str: OAuth token for authentication.
    
    Raises:
        ValueError: If the response does not contain an access token or if there is an HTTP error.
    """
    oauth_url = f"{API_CONFIG['base_oauth_endpoint']}{API_CONFIG['endpoints']['oauth']}"
    client_auth = base64.b64encode(f"{creds['client_id']}:{creds['client_secret']}".encode()).decode()
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Authorization': f'Basic {client_auth}'
    }
    request_body = {"grant_type": "client_credentials"}
    
    response = requests.post(oauth_url, headers=headers, data=request_body)
    response.raise_for_status()  # Raises HTTPError for bad responses
    
    token_response = response.json()
    token = token_response.get('access_token')
    
    if not token:
        raise ValueError("No access_token in OAuth response")
    
    return token


### Function to Invoke Model

This function sends a prompt to the specified model:

-  Constructs the URL for the model invocation.
-  Sets up the request body with the prompt.
-  Configures headers, including the OAuth token and API key.
-  Sends the POST request to invoke the model.
-  Returns the job id if successful.


In [37]:
def invoke_model(prompt, modelId, token, temperature=0):
    """
    Function to invoke a model with a given prompt.
    
    Args:
        prompt (str): The input prompt for the model.
        modelId (str): The ID of the model to be invoked.
        token (str): OAuth token for authentication.
        temperature (int, optional): The temperature setting for the model. Defaults to 0.
    
    Returns:
        str: jobId of the job if the invocation is successful, None otherwise.
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['invoke']}/{modelId}"
    request_body = {"prompt": prompt}
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
        "x-api-key": creds["api_key"]
    }
    
    response = requests.post(url, json=request_body, headers=headers)
    response.raise_for_status()
    
    result = response.json()
    return result.get('jobId')


### Function to Wait for Job Completion

Note that all ModelOps calls are asynchronous

This function checks the status of a job until it is completed or an error occurs:

- Constructs the URL for checking job status.
- Sets up headers with the OAuth token.
- Continuously checks the job status at intervals until completion or timeout.

In [60]:
def wait_for_job_completion(jobId, token):
    """
    Function to wait for the completion of a job.
    
    Args:
        jobId (str): The jobId of the job.
        token (str): OAuth token for authentication.
    
    Returns:
        bool: True if the job is completed successfully, False otherwise.
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['status']}/{jobId}"
    headers = {"Authorization": f"Bearer {token}"}
    total_wait_time = 60
    wait_interval = 10
    
    for _ in range(total_wait_time // wait_interval):
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        result = response.json()
        job_status = result.get('jobStatus')
        
        if job_status in ('Completed', 'Served'):
            return True
        if job_status == 'Error':
            return False
        
        time.sleep(wait_interval)
    
    return False


### Function to Get Job Response

This function retrieves the response of a completed job:

- Constructs the URL for retrieving job data.
- Sets up headers with the OAuth token.
- Sends the GET request to obtain the job response.
- Parses and returns the text content if successful.

In [71]:
def get_response(jobId, token):
    """
    Function to get the response of a completed job.
    
    Args:
        jobId (str): The jobId of the job.
        token (str): OAuth token for authentication.
    
    Returns:
        str: The text content of the job response if successful, None otherwise.
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['data']}/{jobId}/data"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    result = response.json()
    # result = json.loads(result)
    return result
    # return result.get('content', [{}])[0].get('text')


### Example Usage

This section demonstrates how to use the functions together:

- Obtains the OAuth token.
- Invokes the model with a prompt.
- Waits for the job to complete.
- Retrieves and prints the response if successful.

In [80]:
token = get_oauth_token()
token


In [ ]:
prompt = "How many r's are there in the Strawberry?"
modelId = "anthropic-claude-3-haiku"
jobId = invoke_model(prompt, modelId, token)
jobId

In [82]:
completion_success = wait_for_job_completion(jobId , token)
if completion_success:
    resp = get_response(jobId , token) 
else:
    print("error occured")

In [ ]:
resp

In [84]:
def read_image(file_path):
    """
    Function to read an image file and return image_bytes.
    
    Args:
        file_path (str): The path to the image file.
    
    Returns:
        str: image bytes.
    """
    with open(file_path, "rb") as image_file:
        image_bytes = image_file.read()
    return image_bytes

In [85]:
def convert_image_to_base64(image_bytes):
    if image_bytes is None:
        return None
    base_64_encode_data=base64.b64encode(image_bytes)
    return base_64_encode_data.decode('utf-8')

In [86]:
def invoke_with_attachment(base64EncodedData,type,modelId,token):
    """
    Function to invoke a model with attachments.
    
    Returns:
        str: jobId of the job if the invocation is successful, None otherwise.
    """
    url = f"{API_CONFIG['base_url']}{API_CONFIG['endpoints']['invoke']}/{modelId}"
    request_body = {
        "prompt": "Describe given image",
        "attachments": 
            {
                "type": type,
                "base64EncodedData": base64EncodedData
            }
        
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
        "x-api-key": creds["api_key"]
    }
    
    response = requests.post(url, json=request_body, headers=headers)
    response.raise_for_status()
    
    result = response.json()
    return result.get('jobId')


In [ ]:
modelId = "anthropic-claude-3-haiku"
image_bytes=read_image("./drivers_license_image.jpg")
image_bytes
base_64_encoded_data=convert_image_to_base64(image_bytes)
jobId = invoke_with_attachment(base_64_encoded_data,"png",modelId,token)
jobId

In [ ]:
completion_success = wait_for_job_completion(jobId , token)
completion_success

In [ ]:
if completion_success:
    resp = get_response(jobId , token)
    print(resp)
